# 3. SHAP gene rankingRanks genes by their contribution to the classification head using SHAP GradientExplainer (100 background cells, 300 explained cells, training cells only).**Inputs:** trained model, training split**Outputs:** ranked gene list per dataset (`results/<dataset>/fold_*_gpu.json`)**Approximate runtime:** about 1-3 minutes per datasetPaths are set in `scripts/common.py` (`H5_DIR`, `RAW_DIR`, `OUT`). Edit them once before running any notebook.

In [ ]:
import sys, jsonfrom pathlib import Pathsys.path.insert(0, str(Path.cwd().parent / 'scripts'))import common as Cprint('datasets:', list(C.DATASETS))print('results directory:', C.OUT)

In [ ]:
import shap, numpy as npfrom tensorflow.keras.models import Modelfrom ribo_ablation import common_genes, load_densesys.path.insert(0, str(Path.cwd().parent))from train_weights import buildname = 'PC3_high'from sklearn.model_selection import train_test_splitgenes = common_genes(); X, y = load_dense(name, genes)# SHAP uses the training split only (Supplementary Table S6)xtr, xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=4, stratify=y)model = build(X.shape[1]); model.load_weights(f'../weights/{name}_mlcae.weights.h5')head = Model(model.input, model.get_layer('category').output)rng = np.random.default_rng(4)bg = xtr[np.sort(rng.choice(len(xtr), 100, replace=False))]ex = xtr[np.sort(rng.choice(len(xtr), 300, replace=False))]sv = shap.GradientExplainer(head, bg).shap_values(ex)imp = np.abs(sv[1] if isinstance(sv, list) else sv[..., 1]).mean(axis=0)top = np.argsort(-imp)[:10]print('top-10 SHAP genes:', [genes[i] for i in top])